# Customer Churn: Data Quality and Exploratory Analysis

**Dataset note:** This repository contains a small synthetic demonstration dataset because the original Edvyro CSV was not provided in the chat. Replace the raw CSV with the portal file if required.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
DATA = Path('../data')
raw = pd.read_csv(DATA / 'customer_churn_raw.csv')
df = raw.drop_duplicates().copy()
print('Raw shape:', raw.shape, 'Duplicates:', raw.duplicated().sum())
display(df.head())


## Data profile


In [ ]:
profile = pd.DataFrame({'type':raw.dtypes.astype(str),'missing':raw.isna().sum(),'unique':raw.nunique()})
display(profile)
display(raw.describe(include='all').T)


## Cleaning and validation


In [ ]:
for c in ['age','tenure_months','monthly_charges']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df.loc[~df.age.between(18,100),'age'] = np.nan
df.loc[~df.tenure_months.between(0,120),'tenure_months'] = np.nan
df.loc[~df.monthly_charges.between(0,1000),'monthly_charges'] = np.nan
df['contract_type'] = df.contract_type.replace({'month to month':'Month-to-month'})
df['internet_service'] = df.internet_service.replace({'Fiber':'Fiber optic'})
for c in ['age','tenure_months','monthly_charges']:
    df[c] = df[c].fillna(df[c].median())
for c in ['tech_support','payment_method']:
    df[c] = df[c].fillna('Unknown')
assert df.customer_id.is_unique
assert df.isna().sum().sum() == 0
df.to_csv(DATA / 'customer_churn_cleaned.csv', index=False)
print('Cleaned shape:', df.shape)


## Summary statistics and patterns


In [ ]:
display(df.describe(include='all').T)
print('Churn rate (%):', round((df.churn=='Yes').mean()*100,2))
df.churn.value_counts().plot(kind='bar',title='Customer churn counts'); plt.tight_layout(); plt.show()
(df.groupby('contract_type').churn.apply(lambda s:(s=='Yes').mean()*100)).plot(kind='bar',title='Churn rate by contract'); plt.ylabel('Percent'); plt.tight_layout(); plt.show()
df.boxplot(column='monthly_charges',by='churn'); plt.suptitle(''); plt.tight_layout(); plt.show()
